# Exploração — atraso de entrega Olist

Antes de modelar: quanto os pedidos atrasam, **quando**, **onde** e **com quais sinais disponíveis
na hora da compra**. Cada gráfico aqui justifica (ou descarta) uma família de features em
`src/features.py`.

Roteiro: visão geral → atraso no tempo → geografia → prazo prometido → histórico do vendedor →
clima do marketplace → o que isso significa para o modelo.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

from src.config import TARGET
from src.features import build_dataset
from src.plotting import BLUE, INK, INK_2, apply_style

apply_style()
pd.set_option("display.max_columns", None)
PCT = PercentFormatter(1, decimals=0)

In [ ]:
df = build_dataset()

ts = df["order_purchase_timestamp"]
print(f"{len(df):,} pedidos entregues | {ts.min():%Y-%m-%d} a {ts.max():%Y-%m-%d}")
print(f"Taxa de atraso geral: {df[TARGET].mean():.1%}")
df.head()

## 1. Visão geral: uma classe rara

Acurácia não serve como métrica: um modelo que responde "no prazo" para todo pedido já acerta
mais de 90%. A avaliação usa **ROC-AUC / PR-AUC** (qualidade do ranking) e **precisão / recall**
num corte escolhido explicitamente (ver `02_modelagem.ipynb`).

In [ ]:
faltantes = df.isna().mean().sort_values(ascending=False)
faltantes[faltantes > 0].to_frame("% faltante").style.format("{:.2%}")

Os faltantes se concentram nas features de **clima** (os primeiros dias do histórico não têm
entregas anteriores para medir) e em poucos cadastros de produto/CEP. O pipeline imputa pela
mediana / moda dentro do treino.

## 2. O atraso muda muito com o tempo

In [ ]:
mensal = df.set_index("order_purchase_timestamp")[TARGET].resample("MS").agg(["mean", "count"])
mensal = mensal[mensal["count"] >= 100]  # 2016 tem meses com poucas dezenas de pedidos

fig, (ax_taxa, ax_vol) = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True, gridspec_kw={"height_ratios": [1.4, 1]})
ax_taxa.bar(mensal.index, mensal["mean"], width=20, align="edge", color=BLUE)
ax_taxa.axhline(df[TARGET].mean(), color=INK_2, linewidth=1)
ax_taxa.text(mensal.index.min(), df[TARGET].mean() * 1.08, f"média {df[TARGET].mean():.1%}", color=INK_2)
ax_taxa.yaxis.set_major_formatter(PCT)
ax_taxa.set_title("Taxa de atraso por mês de compra")
ax_vol.bar(mensal.index, mensal["count"], width=20, align="edge", color=BLUE)
ax_vol.set_title("Pedidos por mês")
ax_vol.yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f} mil")
for ax in (ax_taxa, ax_vol):
    ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

mensal.assign(mean=lambda m: m["mean"].round(3)).T

**Leitura.**

- A taxa de atraso vai de **1,4% (jun/2018) a 21,4% (mar/2018)**. Os picos coincidem com a
  Black Friday de 2017 (14,3% em novembro) e com fev–mar/2018 (16% e 21%).
- O volume se estabiliza em 6–7 mil pedidos/mês a partir de nov/2017, então os picos não são
  efeito de amostra pequena: são choques da operação.
- O período de teste (os 20% mais recentes, jun–ago/2018) vai de 1,4% a 10,4% em três meses.
  Isso tem duas consequências: a validação **tem** que ser temporal, e um corte fixo de
  probabilidade vai sinalizar volumes muito diferentes de um mês para o outro.
- Daí as features de **clima do marketplace** (`clima_*`, `pressao_demanda`) e de **calendário**
  (`mes_sin/cos`, `dias_para_natal`, `semana_black_friday`).

## 3. Geografia

In [ ]:
por_estado = df.groupby("customer_state")[TARGET].agg(["mean", "count"]).sort_values("mean")

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(por_estado.index, por_estado["mean"], height=0.6, color=BLUE)
ax.axvline(df[TARGET].mean(), color=INK_2, linewidth=1)
for estado, linha in por_estado.tail(5).iterrows():
    ax.text(linha["mean"] + 0.003, estado, f"{linha['mean']:.0%}", va="center", color=INK)
ax.xaxis.set_major_formatter(PCT)
ax.grid(axis="y", visible=False)
ax.set_title("Taxa de atraso por estado do cliente (linha = média)")
plt.tight_layout()
plt.show()

In [ ]:
faixas_km = pd.cut(df["distancia_km"], [0, 100, 300, 600, 1000, 2000, 5000], include_lowest=True)
pd.DataFrame({
    "interestadual": df.groupby("estados_diferentes")[TARGET].mean().rename({0: "mesmo estado", 1: "outro estado"}),
}).T.style.format("{:.1%}")

In [ ]:
por_distancia = df.groupby(faixas_km, observed=True)[TARGET].agg(["mean", "count"])
por_distancia.style.format({"mean": "{:.1%}", "count": "{:,}"})

**Leitura.**

- A distância tem efeito claro e monotônico: **6,4% até 100 km → 13,7% acima de 2.000 km**.
  Pedidos interestaduais atrasam 9,3%, contra 6,1% dentro do mesmo estado.
- O Nordeste lidera (AL 24%, MA 20%, PI 16%), enquanto SP (5,9%) e PR (5,0%) ficam abaixo da média.
- Estados do Norte (RO, AC, AM) atrasam **pouco** apesar da distância: a Olist já promete prazos
  longos para eles. Distância sozinha não basta; ela precisa ser lida junto com o prazo prometido.

## 4. Prazo prometido

A data estimada é mostrada no checkout, então o prazo prometido é uma feature legítima — e a
Olist alarga a estimativa quando espera problemas.

In [ ]:
faixas_prazo = pd.cut(df["prazo_prometido"], [0, 10, 15, 20, 25, 30, 45, 200])
por_prazo = df.groupby(faixas_prazo, observed=True)[TARGET].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(9, 3.6))
rotulos = [f"{int(i.left)}–{int(i.right)}" for i in por_prazo.index]
barras = ax.bar(rotulos, por_prazo["mean"], width=0.6, color=BLUE)
ax.bar_label(barras, labels=[f"{v:.0%}" for v in por_prazo["mean"]], padding=3, color=INK)
ax.yaxis.set_major_formatter(PCT)
ax.grid(axis="x", visible=False)
ax.set_xlabel("prazo prometido (dias)")
ax.set_title("Taxa de atraso por prazo prometido")
plt.tight_layout()
plt.show()

**Leitura.** Prazos de até 10 dias atrasam **14,7%**; acima de 45 dias, **2,0%**. O prazo
prometido carrega a leitura que a própria operação faz do risco da rota.

A relação não é linear: a faixa de até 10 dias é 92% dentro do mesmo estado e mesmo assim
atrasa 14,7% (promessa apertada demais), enquanto a de 10–15 dias (85% no mesmo estado)
atrasa só 5,8%. De 15 dias em diante, mais prazo significa rota mais longa **e** mais folga, e o
atraso cai de forma contínua. Interações assim favorecem modelos de árvore.

## 5. Histórico do vendedor (point-in-time)

`seller_taxa_atraso_historica` usa só pedidos do vendedor **já entregues** na data da compra — o
desfecho de um pedido ainda em trânsito não existia naquele momento. Considerando só vendedores
com pelo menos 10 entregas anteriores:

In [ ]:
com_historico = df[df["seller_pedidos_anteriores"] >= 10]
faixas_hist = pd.cut(com_historico["seller_taxa_atraso_historica"], [-0.001, 0.02, 0.05, 0.1, 0.15, 0.25, 1])
por_hist = com_historico.groupby(faixas_hist, observed=True)[TARGET].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(9, 3.6))
rotulos = ["≤ 2%", "2–5%", "5–10%", "10–15%", "15–25%", "> 25%"][: len(por_hist)]
barras = ax.bar(rotulos, por_hist["mean"], width=0.6, color=BLUE)
ax.bar_label(barras, labels=[f"{v:.0%}" for v in por_hist["mean"]], padding=3, color=INK)
ax.yaxis.set_major_formatter(PCT)
ax.grid(axis="x", visible=False)
ax.set_xlabel("taxa de atraso histórica do vendedor")
ax.set_title("Taxa de atraso do pedido por histórico do vendedor")
plt.tight_layout()
plt.show()
print(f"{len(com_historico) / len(df):.0%} dos pedidos têm vendedor com ≥ 10 entregas anteriores")

**Leitura.** O histórico do vendedor tem sinal, mas moderado: **6,7%** de atraso para quem
atrasou ≤ 2% no passado, contra **~10%** para quem atrasou mais de 5%.

Uma versão anterior desta feature contava todos os pedidos *comprados* antes, incluindo pedidos
ainda em trânsito (7,6% dos pares de pedidos do mesmo vendedor). Isso vazava o futuro: sozinha, a
feature tinha ROC-AUC de 0,616 no treino, contra **0,564** na versão point-in-time. O ganho aparente
vinha do vazamento, não do vendedor.

## 6. Clima do marketplace

`clima_atraso_30d` é a taxa de atraso das entregas concluídas nos 30 dias antes da compra.

In [ ]:
faixas_clima = pd.qcut(df["clima_atraso_30d"], 8)
por_clima = df.groupby(faixas_clima, observed=True)[TARGET].mean()

fig, ax = plt.subplots(figsize=(9, 3.6))
rotulos = [f"{max(i.left, 0):.0%}–{i.right:.0%}" for i in por_clima.index]
ax.bar(rotulos, por_clima.values, width=0.6, color=BLUE)
ax.yaxis.set_major_formatter(PCT)
ax.grid(axis="x", visible=False)
ax.set_xlabel("taxa de atraso do marketplace nos 30 dias anteriores (oitavos)")
ax.set_title("Taxa de atraso do pedido por clima do marketplace")
plt.tight_layout()
plt.show()

**Leitura.** A relação **não é monotônica**. O risco máximo (13,3%) aparece quando o marketplace
atrasou 5–6% no último mês e cai para 5,9% quando esse número passa de 15%.

A média de 30 dias é um sinal **defasado**: logo depois de uma crise ela continua alta enquanto a
operação já se recuperou (abril/2018, depois do pico de março). Por isso o modelo também recebe
`clima_atraso_7d` e `clima_choque` (7 dias ÷ 30 dias), que dizem se a situação está piorando ou
melhorando **agora**. É o tipo de interação que uma árvore aprende e uma regressão linear não.

In [ ]:
dias = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
categorias = df.groupby("product_category_name")[TARGET].agg(["mean", "count"])
display(df.groupby("dia_semana_compra")[TARGET].mean().set_axis(dias).to_frame("taxa de atraso").T.style.format("{:.1%}"))
categorias[categorias["count"] >= 500].sort_values("mean", ascending=False).head(8).style.format({"mean": "{:.1%}", "count": "{:,}"})

**Dia da semana e categoria** têm efeito pequeno (7,5%–9,1% entre dias; 5,5%–9,9% entre as
categorias com ≥ 500 pedidos). Entram no modelo, mas não são o sinal principal.

## Conclusões para a modelagem

1. **Classe rara (8,1%)**: avaliar por ROC-AUC/PR-AUC e escolher o corte explicitamente;
   acurácia e o corte padrão de 0.5 enganam.
2. **Taxa de atraso instável no tempo**: split e validação **temporais**. O teste (jun–ago/2018)
   tem taxa de atraso diferente do treino (5,3% contra 8,8%).
3. **Sinais mais fortes**: distância/rota, prazo prometido, clima recente do marketplace e
   calendário (Black Friday, fim de ano).
4. **Relações não lineares e com interação** (clima, prazo): modelo de árvores (XGBoost).
5. **Toda feature histórica é point-in-time**: só entram desfechos já conhecidos na data da compra.